# get similarity and coherence

In [1]:
import os
import re
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_distances

In [2]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [3]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(study1_dir + model_path, binary=True)

## 데이터셋 준비

In [4]:
tbl_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', encoding='ISO-8859-1')
tbl_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,friend21,friend22,friend23,friend24,friend25,friend26,friend27,friend28,friend29,friend30
0,1,card,bank,money,green,yellow,blue,eye,nose,smell,...,up,down,below,NaN,wine,drink,alcohol,rum,beer,cider
1,2,lock,door,big,lion,roar,scary,ghost,dark,halloween,...,blue,eyes,see,view,countryside,beauty,landscape,mountain,big,dog
2,3,door,window,curtain,draft,cold,snow,scarf,wooly hat,winter,...,high,beam,wood,forest,trees,green,grass,wildlife,nature,climate


In [5]:
seed_words = ['key', 'money', 'friend']
target_words = ['money', 'friend']

n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(tbl_data) # 210
n_dim_of_vector = 300

In [6]:
for seed_word in seed_words: # key, money, friend
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # key1~30, money1~30, friend1~30

    for column in word_columns:
        # vector field 생성
        tbl_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # 피험자 한 명의 응답 단어들 벡터 처리
        for i_subject in range(n_subject):
            try:
                response_word = tbl_data.iloc[i_subject][column]
                if pd.isna(response_word) or len(response_word.strip()) == 0:# NaN, 값이 빈 칸 & 응답안해서 '', ' '로 저장된 경우 걸러내기
                    tbl_data[column + '_vec'][i_subject] = None
                    continue

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]
                    if len(response_word) == 0: # 앞에서 걸러져서 결과가 없으면, 넘어가기
                        tbl_data[column + '_vec'][i_subject] = None
                        continue

                    vec_word2vec = np.zeros((n_dim_of_vector, 0))  # 300차원의 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        # reshape: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기
                    # 각 열(단어 벡터)에 대한 평균 계산
                    average_vector = np.mean(vec_word2vec, axis=1)
                    tbl_data[column + '_vec'][i_subject] = average_vector
            except:
                pass

/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_19024/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_19024/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_19024/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/us

# similarity - coherence

- 피험자가 응답한 모든 단어 ↔ `money` 간의 similarity 
- 피험자가 응답한 모든 단어 ↔ `friend` 간의 similarity \
\
\
-> coherence값 구함

In [7]:
# similarity 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)]
        for column in word_columns: 
                column_name = f'similarity_{column}_{target_word}'
                tbl_data = tbl_data.assign(**{column_name: None})
# coherence 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        column_name = f'coherence_{seed_word}_{target_word}'
        tbl_data = tbl_data.assign(**{column_name: None})

    # coherence 평균
    avg_column_name = f'coherence_{target_word}'
    tbl_data = tbl_data.assign(**{avg_column_name: None})
tbl_data.columns

Index(['subject', 'key1', 'key2', 'key3', 'key4', 'key5', 'key6', 'key7',
       'key8', 'key9',
       ...
       'similarity_friend29_friend', 'similarity_friend30_friend',
       'coherence_key_money', 'coherence_money_money',
       'coherence_friend_money', 'coherence_money', 'coherence_key_friend',
       'coherence_money_friend', 'coherence_friend_friend',
       'coherence_friend'],
      dtype='object', length=369)

In [8]:
# 응답 단어 컬럼 목록
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
# word_columns

In [9]:
vectors_data = tbl_data.iloc[:, 91:181]
vectors_data[0:3]

,key1_vec,key2_vec,key3_vec,key4_vec,key5_vec,key6_vec,key7_vec,key8_vec,key9_vec,key10_vec,...,friend21_vec,friend22_vec,friend23_vec,friend24_vec,friend25_vec,friend26_vec,friend27_vec,friend28_vec,friend29_vec,friend30_vec
0,"[-0.1630859375, 0.1435546875, 0.197265625, 0.1...","[0.02197265625, 0.134765625, -0.057861328125, ...","[0.158203125, 0.05126953125, 0.06640625, 0.210...","[0.05908203125, 0.2021484375, 0.1796875, -0.04...","[-0.0732421875, 0.0263671875, 0.076171875, 0.1...","[0.0390625, 0.08642578125, 0.2236328125, 0.118...","[0.162109375, 0.08642578125, -0.033447265625, ...","[0.0201416015625, 0.10498046875, 0.07080078125...","[0.1884765625, -0.07763671875, -0.042724609375...","[-0.05224609375, 0.22265625, 0.057861328125, 0...",...,"[0.1201171875, -0.0201416015625, 0.20703125, 0...","[0.0245361328125, -0.10498046875, 0.26171875, ...","[-0.1796875, -0.412109375, -0.057861328125, 0....",None,"[-0.17578125, -0.109375, -0.189453125, 0.15625...","[-0.13671875, -0.27734375, 0.04736328125, 0.22...","[-0.0615234375, -0.337890625, 0.054443359375, ...","[-0.3515625, -0.1796875, -0.06787109375, 0.146...","[-0.08740234375, -0.1337890625, -0.044921875, ...","[-0.21875, -0.1533203125, -0.23828125, 0.33789..."
1,"[0.0179443359375, 0.193359375, -0.06298828125,...","[0.09619140625, -0.026611328125, 0.0791015625,...","[0.111328125, 0.10595703125, -0.07373046875, 0...","[0.212890625, -0.00457763671875, -0.236328125,...","[0.349609375, 0.0595703125, -0.0057373046875, ...","[0.171875, -0.1240234375, 0.1748046875, 0.1601...","[0.2265625, 0.15625, 0.0031280517578125, -0.10...","[0.12109375, 0.1455078125, 0.1455078125, -0.20...","[-0.058349609375, 0.038330078125, 0.0245361328...","[0.0712890625, 0.306640625, -0.15234375, 0.398...",...,"[0.0390625, 0.08642578125, 0.2236328125, 0.118...","[0.208984375, 0.1279296875, 0.283203125, -0.06...","[-0.0556640625, 0.0089111328125, -0.0922851562...","[-0.0703125, -0.0244140625, -0.1884765625, 0.1...","[-0.045654296875, 0.31640625, 0.03515625, 0.01...","[-0.004669189453125, 0.2412109375, 0.068359375...","[0.10595703125, 0.24609375, -0.130859375, -0.1...","[0.154296875, 0.08349609375, -0.044921875, 0.0...","[0.111328125, 0.10595703125, -0.07373046875, 0...","[0.05126953125, -0.0223388671875, -0.172851562..."
2,"[0.09619140625, -0.026611328125, 0.0791015625,...","[0.0986328125, 0.08642578125, -0.07763671875, ...","[0.2216796875, 0.146484375, 0.1484375, -0.2353...","[0.0126953125, 0.396484375, 0.2421875, 0.20117...","[-0.0189208984375, 0.11865234375, -0.0625, 0.0...","[-0.01239013671875, -0.138671875, -0.019409179...","[0.040283203125, -0.08251953125, 0.1796875, 0....","[-0.05474853515625, 0.05194091796875, -0.20458...","[-0.058837890625, 0.337890625, -0.1376953125, ...","[-0.1376953125, 0.20703125, -0.2578125, -0.001...",...,"[0.07666015625, 0.00970458984375, -0.080078125...","[-0.2294921875, 0.2001953125, -0.0654296875, 0...","[0.029052734375, 0.2373046875, -0.08935546875,...","[0.337890625, 0.1708984375, -0.002838134765625...","[0.49609375, 0.22265625, 0.01312255859375, 0.1...","[0.05908203125, 0.2021484375, 0.1796875, -0.04...","[0.1025390625, 0.2275390625, 0.1318359375, 0.1...","[-0.0703125, 0.2021484375, -0.1552734375, 0.03...","[0.138671875, 0.2041015625, 0.0289306640625, 0...","[0.1787109375, 0.326171875, -0.11865234375, 0...."


In [10]:
for i_subject in range(n_subject):

    for target_word in target_words: # money, friend
        target_word_vec = word2vec_model[target_word]
        coherences_per_sub = []

        for seed_word in seed_words:  # key, money, friend
            # 각 피험자의 응답 단어들을 30개씩 단어 리스트로 변환
            seed_response_words = [vectors_data.iloc[i_subject][f'{column}_vec'] for column in word_columns if column.startswith(seed_word)]
            # nan값 걸러내기
            seed_response_words = [word for word in seed_response_words if not isinstance(word, float) or not np.isnan(word) or word  == ' ' or word  == ''] 

            if len(seed_response_words) > 0: # 애초에 값이 0인 피험자 데이터들은 건너뛰도록 함
                each_seed_similarities = []

                for i_word, response_word in enumerate(seed_response_words): # 각각 seed1~30
                    try:
                        # 구(phrase) 벡터와 타겟 단어 벡터 사이의 코사인 거리 계산
                        cosine_distance = cosine_distances(response_word.reshape(1, -1), target_word_vec.reshape(1, -1))
                        # 코사인 거리를 유사도로 변환
                        cosine_similarity = 1 - cosine_distance
                        # print(f'subject: {i_subject}, seed: {seed_word}, target: {target_word}, response:{i_word}, {response_word}, sim: {similarity_with_target}')

                        # 추출한 유사도를 해당 테이블 위치에 저장
                        tbl_data.at[i_subject, f'similarity_{seed_word}{i_word+1}_{target_word}'] = cosine_similarity[0][0]
                        # coherence을 구하기 위해 배열에 저장
                        each_seed_similarities.append(cosine_similarity) # 30개

                    except Exception as e:
                        print(f'subject {i_subject+1}의 {seed_word}{i_word+1}와 {target_word} 단어 유사도 계산에 실패했습니다.')
                        continue

                # 해당 피험자가 하나의 seed에 답한 40개의 응답단어의 유사도의 coherence값
                coherence_of_seed_per_sub = np.mean(each_seed_similarities) 
                # print('coherence: ',coherence_of_seed_per_sub)

                # 해당 피험자에 대한, 그 seed단어 각각(4개)에 대한 coherence값
                # 피험자마자 6개값( key,money,friend - money,friend 조합)
                coherences_per_sub.append(coherence_of_seed_per_sub)

                # # Append the coherence values to the data frame for the current subject
                tbl_data.at[i_subject, f'coherence_{seed_word}_{target_word}'] = coherence_of_seed_per_sub
                tbl_data.at[i_subject, f'coherence_{target_word}'] = sum(coherences_per_sub) / len(coherences_per_sub)


subject 1의 friend24와 money 단어 유사도 계산에 실패했습니다.
subject 1의 friend24와 friend 단어 유사도 계산에 실패했습니다.
subject 22의 money13와 money 단어 유사도 계산에 실패했습니다.
subject 22의 friend13와 money 단어 유사도 계산에 실패했습니다.
subject 22의 money13와 friend 단어 유사도 계산에 실패했습니다.
subject 22의 friend13와 friend 단어 유사도 계산에 실패했습니다.
subject 31의 friend6와 money 단어 유사도 계산에 실패했습니다.
subject 31의 friend11와 money 단어 유사도 계산에 실패했습니다.
subject 31의 friend6와 friend 단어 유사도 계산에 실패했습니다.
subject 31의 friend11와 friend 단어 유사도 계산에 실패했습니다.
subject 40의 friend18와 money 단어 유사도 계산에 실패했습니다.
subject 40의 friend18와 friend 단어 유사도 계산에 실패했습니다.
subject 67의 friend1와 money 단어 유사도 계산에 실패했습니다.
subject 67의 friend1와 friend 단어 유사도 계산에 실패했습니다.
subject 68의 friend8와 money 단어 유사도 계산에 실패했습니다.
subject 68의 friend8와 friend 단어 유사도 계산에 실패했습니다.
subject 69의 money19와 money 단어 유사도 계산에 실패했습니다.
subject 69의 money19와 friend 단어 유사도 계산에 실패했습니다.
subject 74의 key27와 money 단어 유사도 계산에 실패했습니다.
subject 74의 key28와 money 단어 유사도 계산에 실패했습니다.
subject 74의 key27와 friend 단어 유사도 계산에 실패했습니다.
subject 74의 key28와 frien

In [12]:
# vector 컬럼들 드롭 ( csv용량이 너무 커지는 것을 방지 )
drop_columns = tbl_data.columns[91:181]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'similarity_coherence_with_words.csv', index=None)

In [13]:
# 단어 컬럼들 드롭
drop_columns = tbl_data.columns[1:91]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

# 단어 없이 coherence만 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'similarity_coherence.csv', index=None)